# Omega-7 Voice Training
Trains a Piper VITS voice model from the ElevenLabs-generated corpus.

**Before running:**
1. Upload your `voice_training/` folder to Google Drive (My Drive root)
2. Set Runtime → Change runtime type → **T4 GPU**
3. Run all cells in order

**Output:** `omega7.onnx` + `omega7.onnx.json` saved back to Drive.
Copy both files to `models/` in the skull project and set `PIPER_MODEL_PATH=models/omega7.onnx` in `.env`.

In [ ]:
# ── 1. Verify GPU ─────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU detected. Go to Runtime → Change runtime type → T4 GPU')
print(result.stdout)

In [ ]:
# ── 2. Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE    = pathlib.Path('/content/drive/MyDrive')
DATA_DIR = DRIVE / 'voice_training'
PRE_DIR  = DRIVE / 'voice_training' / 'preprocessed'
MODEL_OUT = DRIVE / 'voice_training' / 'omega7.onnx'

assert DATA_DIR.exists(), (
    'voice_training/ not found in My Drive.\n'
    'Upload the folder from your skull project and re-run this cell.'
)
wavs = list((DATA_DIR / 'wavs').glob('*.wav'))
print(f'Found {len(wavs)} WAV files')
print(f'Metadata: {DATA_DIR / "metadata.csv"}')

In [ ]:
# ── 3. Install piper training stack ──────────────────────────────────────────
# piper-phonemize is NOT on PyPI — must be fetched from GitHub releases.
# espeak-ng is its system-level dependency.
import sys, subprocess

print('Installing system dependency: espeak-ng')
subprocess.run(['apt-get', 'install', '-y', '-q', 'espeak-ng', 'libespeak-ng-dev'], check=True)

# Build the wheel URL for the current Python version (Colab is Linux x86_64)
py  = f"{sys.version_info.major}{sys.version_info.minor}"
tag = f"cp{py}"
wheel_url = (
    f"https://github.com/rhasspy/piper-phonemize/releases/download/v1.1.0/"
    f"piper_phonemize-1.1.0-{tag}-{tag}-linux_x86_64.whl"
)
print(f'Python {sys.version_info.major}.{sys.version_info.minor} detected')
print(f'Fetching piper-phonemize wheel from:\n  {wheel_url}')

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', wheel_url],
    capture_output=True, text=True
)
if result.returncode != 0:
    # Wheel may not exist for this Python version — try building from source
    print('Pre-built wheel not found, building piper-phonemize from source...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/rhasspy/piper-phonemize.git',
                    '/content/piper-phonemize'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    '/content/piper-phonemize'], check=True)
else:
    print('piper-phonemize installed from wheel')

print('Cloning piper repo...')
subprocess.run(['git', 'clone', '--depth', '1',
                'https://github.com/rhasspy/piper.git', '/content/piper'], check=True)

print('Installing piper training deps...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-e', '/content/piper/src/python[train]'], check=True)

print('\nAll dependencies installed.')

In [ ]:
# ── 4. Verify install ─────────────────────────────────────────────────────────
import subprocess, sys
result = subprocess.run([sys.executable, '-c', 'import piper_train; print("piper_train OK")'],
                        capture_output=True, text=True)
print(result.stdout or result.stderr)

import pathlib
training_md = pathlib.Path('/content/piper/TRAINING.md')
if training_md.exists():
    print(training_md.read_text())
else:
    print('TRAINING.md not found — check /content/piper/ for docs')

In [ ]:
# ── 5. Preprocess dataset ─────────────────────────────────────────────────────
import pathlib, subprocess, sys
DRIVE     = pathlib.Path('/content/drive/MyDrive')
DATA_DIR  = DRIVE / 'voice_training'
PRE_DIR   = DRIVE / 'voice_training' / 'preprocessed'
PRE_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run([
    sys.executable, '-m', 'piper_train.preprocess',
    '--language',      'en-us',
    '--input-dir',     str(DATA_DIR),
    '--output-dir',    str(PRE_DIR),
    '--single-speaker',
    '--sample-rate',   '22050',
], check=True)

print('Preprocessing complete.')
for p in sorted(PRE_DIR.iterdir()):
    print(f'  {p.name}')

In [ ]:
# ── 6. Train ──────────────────────────────────────────────────────────────────
# ~4–8 hours on a T4 for 6000 epochs with ~500 sentences.
# Checkpoints save continuously — re-run this cell to resume after a session drop.
import pathlib, subprocess, sys
DRIVE    = pathlib.Path('/content/drive/MyDrive')
PRE_DIR  = DRIVE / 'voice_training' / 'preprocessed'
CKPT_DIR = PRE_DIR / 'lightning_logs' / 'version_0' / 'checkpoints'

existing = sorted(CKPT_DIR.glob('*.ckpt')) if CKPT_DIR.exists() else []
cmd = [
    sys.executable, '-m', 'piper_train',
    '--dataset-dir',    str(PRE_DIR),
    '--accelerator',    'gpu',
    '--devices',        '1',
    '--quality',        'medium',
    '--batch-size',     '32',
    '--validation-split', '0.05',
    '--max_epochs',     '6000',
]
if existing:
    cmd += ['--resume_from_checkpoint', str(existing[-1])]
    print(f'Resuming from: {existing[-1].name}')
else:
    print('Starting fresh training run')

subprocess.run(cmd, check=True)

In [ ]:
# ── 7. Export to ONNX ─────────────────────────────────────────────────────────
import pathlib, subprocess, sys
DRIVE     = pathlib.Path('/content/drive/MyDrive')
PRE_DIR   = DRIVE / 'voice_training' / 'preprocessed'
MODEL_OUT = DRIVE / 'voice_training' / 'omega7.onnx'
CKPT_DIR  = PRE_DIR / 'lightning_logs' / 'version_0' / 'checkpoints'

checkpoints = sorted(CKPT_DIR.glob('*.ckpt'))
if not checkpoints:
    raise FileNotFoundError(f'No checkpoints found in {CKPT_DIR}')

best = checkpoints[-1]
print(f'Exporting: {best.name}')

subprocess.run([
    sys.executable, '-m', 'piper_train.export_onnx',
    '--checkpoint', str(best),
    '--output',     str(MODEL_OUT),
], check=True)

for out in [MODEL_OUT, MODEL_OUT.with_suffix('.onnx.json')]:
    if out.exists():
        mb = out.stat().st_size / 1_000_000
        print(f'  {out.name}  ({mb:.1f} MB)')
    else:
        print(f'  WARNING: {out.name} not found')

In [ ]:
# ── 8. Done ───────────────────────────────────────────────────────────────────
print('Training complete.')
print()
print('Download from Google Drive:')
print('  MyDrive/voice_training/omega7.onnx')
print('  MyDrive/voice_training/omega7.onnx.json')
print()
print('Copy both to your skull project: models/')
print()
print('Set in .env:')
print('  PIPER_MODEL_PATH=models/omega7.onnx')
print('  TTS_BACKEND=piper')